In [0]:
from pyspark.sql.functions import (
    col,
    to_timestamp,
    window,
    count
)

In [0]:
df_orders_stream = (
    spark.readStream
    .table("olist_ecommerce.bronze.orders")
)
df_orders_stream.printSchema()

In [0]:
df_orders_stream = (
    df_orders_stream
    .withColumn(
        "event_time",
        to_timestamp("order_purchase_timestamp")
    )
)

In [0]:
df_orders_stream = (
    df_orders_stream
    .withWatermark(
        "event_time",
        "2 hours"
    )
)

In [0]:
df_hourly_orders = (
    df_orders_stream
    .groupBy(
        window(
            col("event_time"),
            "1 hour"
        )
    )
    .agg(
        count("order_id").alias("total_orders")
    )
)

In [0]:
checkpoint_path = (
    "/Volumes/olist_ecommerce/bronze/checkpoints/orders_stream"
)

In [0]:
%sql

CREATE VOLUME IF NOT EXISTS olist_ecommerce.bronze.checkpoints;

In [0]:
query = (
    df_hourly_orders
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        "olist_ecommerce.gold.streaming_hourly_orders"
    )
)

In [0]:
%sql

SELECT *
FROM olist_ecommerce.gold.streaming_hourly_orders
ORDER BY window.start
LIMIT 50;

In [0]:
%sql

CREATE TABLE IF NOT EXISTS olist_ecommerce.bronze.orders_stream_source
USING DELTA
AS

SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp
FROM olist_ecommerce.bronze.orders
WHERE 1 = 0;

In [0]:
%sql

SELECT COUNT(*) AS total
FROM olist_ecommerce.bronze.orders_stream_source;

In [0]:
%sql

INSERT INTO olist_ecommerce.bronze.orders_stream_source

SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp
FROM olist_ecommerce.bronze.orders
LIMIT 10;

In [0]:
%sql

SELECT *
FROM olist_ecommerce.bronze.orders_stream_source;

In [0]:
from pyspark.sql.functions import col, to_timestamp, window, count

df_orders_stream = (
    spark.readStream
    .table("olist_ecommerce.bronze.orders_stream_source")
    .withColumn(
        "event_time",
        to_timestamp("order_purchase_timestamp")
    )
    .withWatermark(
        "event_time",
        "2 hours"
    )
)

In [0]:
df_hourly_orders = (
    df_orders_stream
    .groupBy(
        window(
            col("event_time"),
            "1 hour"
        )
    )
    .agg(
        count("order_id").alias("total_orders")
    )
)

In [0]:
checkpoint_path = (
    "/Volumes/olist_ecommerce/bronze/checkpoints/orders_stream_demo"
)

In [0]:
query = (
    df_hourly_orders
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(
        availableNow=True
    )
    .toTable(
        "olist_ecommerce.gold.streaming_demo_hourly_orders"
    )
)

In [0]:
%sql

SELECT *
FROM olist_ecommerce.gold.streaming_demo_hourly_orders
ORDER BY window.start;